# Label Smoothing Loss

**难度:** Easy

实现标签平滑交叉熵损失。

标签平滑通过将 one-hot 目标分布与均匀分布混合来防止过度自信，在 Transformer 训练中被广泛使用。

**签名:** `label_smoothing(logits, targets, smoothing=0.1) -> Tensor`

**参数:**
- `logits` — 模型原始输出，形状 `(N, C)`
- `targets` — 整数类别索引，形状 `(N,)`
- `smoothing` — 平滑系数 ε ∈ [0, 1)

**返回:** 标量损失

**公式:** 正确类别的软目标 = `1 - ε`，其他类别 = `ε / (C - 1)`

**提示:**
1. `soft = full((N,C), ε/(C-1))`，再 `soft.scatter_(1, targets, 1-ε)`
2. 用稳定 log-softmax 得到 `log_probs`
3. `return -(soft * log_probs).sum(dim=-1).mean()`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写标签平滑交叉熵 ----
# 面试要点：理解标签平滑的数学含义 + scatter_ 操作

def label_smoothing(logits, targets, smoothing=0.1):
    N, C = logits.shape

    # 面试考点：为什么不用 log(softmax(x)) 而用 log_softmax(x)？
    # 答：当 x 很大时，exp(x) 溢出 → softmax 得到 nan → log(nan) = nan
    # log_softmax 内部用 log-sum-exp 技巧：
    # log(softmax(x_i)) = x_i - log(Σ exp(x_j)) = x_i - max(x) - log(Σ exp(x_j - max(x)))
    # 减去 max 后 exp 不会溢出
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # 面试考点：标签平滑的数学含义
    # 原始 one-hot: [0, 0, 1, 0] (target=2, C=4)
    # 平滑后: [ε/3, ε/3, 1-ε+ε/3, ε/3] = [ε/3, ε/3, 1-2ε/3, ε/3]
    # 简化：非目标 = ε/(C-1)，目标 = 1-ε
    # 验证：(C-1)*ε/(C-1) + (1-ε) = ε + 1-ε = 1 ✓
    soft_targets = torch.full_like(log_probs, smoothing / (C - 1))
    soft_targets.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)

    # 交叉熵：-Σ p·log(q)
    # 当 p 是 one-hot 时退化为 -log(q_target)
    # 当 p 是软目标时，也惩罚"对其他类别概率太低"的行为
    return -(soft_targets * log_probs).sum(dim=-1).mean()

In [ ]:
from torch_judge import check
check("label_smoothing")

## 📝 核心思路总结

1. **标签平滑 = 混合 one-hot 和均匀分布**：防止模型对正确类别过度自信（overconfidence）
2. **scatter_ 实现高效赋值**：在目标位置写入 1-ε，其余位置已经是 ε/(C-1)
3. **交叉熵 = -Σ p·log(q)**：用软目标 p 和模型 log 概率 q 计算 KL 散度